# Cómo está armado este banco

Un Arduino UNO, un sensor magnético de ángulo, un puente en H y un motor. Este
notebook recorre el hardware: qué está conectado a qué, por qué está conectado
así, y qué se puede medir con cada parte --incluso mientras alguna de esas partes
todavía no está sobre la mesa.

El banco se arma en cuatro pasos, y **cada paso ya sirve para algo**:

| | Configuración | Qué se agrega | Qué habilita |
|---|---|---|---|
| **A** | sólo el sensor | AS5600 en el bus I2C | el ángulo, el desenrollado, los filtros, el ritmo del lazo |
| **B** | **con** puente en H, sección 2.1 | L298N y su fuente | mover el motor en los dos sentidos |
| **B′** | **con** actuador de un solo cuadrante, sección 2.2 | un transistor y un diodo, o el mismo puente cableado para un lado | lo mismo, empujando para un solo lado |
| **C** | **sin** sensor de corriente | -- | todo lo anterior; `i` queda como un canal que no mide |
| **D** | **con** ACS712 | el sensor de corriente en A0 | ver el consumo, sobre todo el de arranque&nbsp;⁽¹⁾ |

**B′ es probablemente el caso más común**, y no es un banco de segunda: la
identificación de la planta sale completa con un solo cuadrante. La sección 2.2 dice
qué cambia, y es corta.

⁽¹⁾ Con reservas: la medición de corriente de este banco **no funciona del todo
bien**. La nota al pie de la sección 4 dice exactamente por qué, y no es un cable
flojo: es una cadena de decisiones razonables que no cierran entre sí.

Después del recorrido hay dos secciones de código: la **API** con la que se maneja
todo esto desde Python, y un **punto de partida para identificar la planta**, que
es de donde conviene arrancar la práctica.

Dos avisos antes de correr nada:

> ⚠️ **Con el motor conectado, el motor se mueve.** Revisar que el eje esté libre.

> Hay **un solo `dev`**, el de la celda que sigue, y el notebook se recorre de
> arriba abajo. Si se toca el cableado en
> el medio, volver a correr esa celda.

In [ ]:
import sys, os
sys.path.insert(0, '../python')

import numpy as np
import matplotlib.pyplot as plt

import ensayo
from banco_simulado import conseguir_banco

# --- cómo se ven los gráficos de este notebook -------------------------------
AZUL, NARANJA, AQUA, AMARILLO = '#2a78d6', '#eb6834', '#1baf7a', '#eda100'
TINTA, TENUE, NUBE = '#0b0b0b', '#52514e', '#c9c8c3'

plt.rcParams.update({
    'figure.figsize': (9, 3.6), 'figure.dpi': 110,
    'axes.grid': True, 'axes.axisbelow': True, 'grid.color': '#e6e5e1',
    'grid.linewidth': 0.8, 'axes.edgecolor': '#c9c8c3', 'axes.linewidth': 0.8,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.labelcolor': TENUE, 'axes.titlecolor': TINTA, 'axes.titlelocation': 'left',
    'axes.titleweight': 'medium', 'axes.titlepad': 10,
    'xtick.color': TENUE, 'ytick.color': TENUE, 'text.color': TINTA,
    'lines.linewidth': 1.8, 'legend.frameon': False, 'font.size': 10,
})

# El banco de verdad si el cable está enchufado; si no, uno simulado que lo dice.
dev = conseguir_banco(forzar_simulado=os.environ.get('HW_SIMULADO') == '1')
SIMULADO = getattr(dev, 'simulado', False)

# La calibración del sensor de este banco, si está medida. No es tema de este
# notebook --se mide en calibracion.ipynb-- pero sin ella el ángulo llega torcido,
# y una identificación de velocidad hereda la ondulación como si fuera del motor.
if not SIMULADO:
    try:
        import calib
        from bench import CALIBRACION
        if CALIBRACION.exists():
            calib.asegurar(dev, CALIBRACION)
            print('calibracion del sensor aplicada')
        else:
            print('sin calibracion del sensor: el angulo va crudo '
                  '(ver notebooks/calibracion.ipynb)')
    except Exception as exc:
        print(f'no se pudo aplicar la calibracion: {exc}')


## 0. El mapa

En el medio hay un UNO haciendo tres cosas a la vez, y conviene tenerlas separadas
en la cabeza porque cada una tiene su propio reloj:

1. **Muestrea el sensor a 5 kHz**, con el Timer2, período rígido. Una lectura del
   AS5600 por interrupción, sin esperar activamente.
2. **Pone el comando sobre el puente a 500 Hz**: cada `tickdiv` muestras --10 por
   omisión-- lee la corriente, desenrolla el ángulo y escribe el PWM.
3. **Emite telemetría a 1 Mbaud**: una fila por período, 25 bytes, el 13 % del
   enlace. Es lo que llega a este notebook como un `DataFrame`.

```
   ┌────────────────────┐
   │    Arduino UNO     │   A4/A5 ──── I2C ─────►  AS5600      (configuración A)
   │                    │
   │  Timer2 → 5 kHz    │   9/6/7 ──── PWM+dir ─►  L298N ─► motor   (B)
   │  filas  → 500 Hz   │
   │  USART  → 1 Mbaud  │   A0    ◄─── analógica   ACS712     (C y D)
   └────────┬───────────┘
            │ USB
            ▼
        Jupyter
```

Los pines, completos:

| Señal | Pin | Hace falta para | Si falta |
|---|---|---|---|
| AS5600 `SDA` | A4 | el ángulo | el ángulo queda congelado, el muestreo mantiene su período |
| AS5600 `SCL` | A5 | ídem | ídem |
| AS5600 `VDD` / `GND` | 5V / GND | ídem | ídem |
| L298N `ENA` | 9 (PWM) | mover el motor | se puede medir todo lo que no requiera movimiento |
| L298N `IN1` | 6 | el sentido | el puente empuja siempre para el mismo lado |
| L298N `IN2` | 7 | ídem | ídem |
| Salida del ACS712 | A0 | la corriente | `i` informa ceros, y `bringup()` lo detecta |

**Periféricos de los que el sketch se apropia.** Vale saberlo antes de agregarle
algo al montaje, porque las funciones de Arduino que uno esperaría no están
disponibles:

| Recurso | Para qué | Qué deja de funcionar |
|---|---|---|
| Timer2 | el muestreador de 5 kHz | `analogWrite()` en 3 y 11, `tone()` |
| Timer1 | el PWM del puente, con su propio TOP | `analogWrite()` en 9 y 10, `Servo` |
| ADC | se maneja a mano, canal 0 | `analogRead()` |
| USART | 1 Mbaud, el protocolo | `Serial` para cualquier otra cosa |
| TWI | `nI2C`, por interrupciones | `Wire` |
| Timer0 | -- | nada: `millis()` y el PWM de 5 y 6 andan como siempre (acá el 6 lo usa `IN1`, como salida digital) |

## 1. Configuración A: sólo el sensor

Sin puente, sin motor y sin medición de corriente. Es la configuración con la que
conviene empezar, y no es un juguete: acá ya se ve el sensor, el desenrollado, el
ruido y el ritmo del muestreo. Lo único que falta es que algo gire solo.

```
   ┌───────────────┐                            ┌─────────────┐
   │  Arduino UNO  │  A4 ──── SDA ──────────────│   AS5600    │
   │               │  A5 ──── SCL ──────────────│  (plaqueta) │
   │  Timer2:5 kHz │  5V ──── VDD ──────────────│             │
   │  USB: 1 Mbaud │ GND ──── GND ──────────────│             │
   └───────────────┘                            └──────┬──────┘
                                                       │
                                                imán diametral
                                             1-2 mm sobre el chip
```

**El imán es parte del sensor, no un accesorio.** Tiene que estar magnetizado
**diametralmente** --los polos enfrentados a lo ancho, no norte arriba y sur
abajo--, girar sobre la cara del chip a un par de milímetros, y estar centrado con
el eje de giro. La hoja de datos pide un cuarto de milímetro de excentricidad; lo
que sobra de eso no aparece como ruido sino como una función fija del ángulo que se
repite vuelta tras vuelta. Eso es el tema completo de `calibracion.ipynb`.

**Las plaquetas comerciales de AS5600 traen su propio regulador y los pull-ups del
bus**, así que se enchufan a 5 V y andan. Un chip pelado en modo 3,3 V necesita
adaptación de niveles y sus propias resistencias de pull-up.

**El AS5600 tiene un filtro adentro**, y viene mal puesto para esto: arranca en
16x, que son 2,2 ms de retardo. El sketch lo pasa a 2x, 0,286 ms, al arrancar. Un
retardo en el sensor se identifica después como si fuera del motor, así que
conviene que sea el menor posible y saber cuánto es.

**El muestreo no se detiene si el sensor no está.** Si el AS5600 no contesta, el
muestreador pasa a sondear el bus dos veces por segundo en lugar de cinco mil, y
todo lo demás --el período, la telemetría, los parámetros-- sigue igual. Sirve para
probar la cadena completa (compilar, grabar, capturar, graficar) antes de tener el
sensor sobre la mesa.

`bringup()` es la verificación del equipo, subsistema por subsistema. Con
`motor=False` saltea todo lo que haría girar el eje, que es exactamente lo que
corresponde en esta configuración.

In [ ]:
dev.bringup(motor=False)

Lo que hay que mirar en esa salida, en orden:

- **`muestreo`**: la frecuencia real contra la nominal, medida con el reloj
  de esta computadora y no con el contador de la placa --que avanza una vez por
  período *atendido* y por eso daría siempre por bueno lo que hay que detectar.
- **`margen de tiempo`**: cuánto del período se consume en el peor caso. Con la
  tabla de canales por omisión, a 500 Hz, ronda el 30 %, casi todo en emitir la
  fila. Sin margen alguno se están por empezar a perder períodos.
- **`iman`**: lo que el propio AS5600 dice de su imán: detectado, muy débil o muy
  fuerte. En los dos extremos el imán está a la distancia equivocada, y ahí no
  hay calibración que arregle nada.
- **`bus i2c`**: errores de transferencia y desbordes. Un puñado de desbordes por
  segundo es normal --el diagnóstico del imán lee un registro extra dos veces por
  segundo y esa lectura no entra en 200 us--; lo que no es normal es que el bus no
  llegue de manera sostenida.

Y ahora el sensor en vivo. Hay que **girar el imán con la mano** mientras la celda
corre. `y_uw` es el ángulo *desenrollado*: sigue contando a través de la vuelta de
4096 cuentas en lugar de saltar a cero, así que un eje que gira sin parar da una
recta que sube sin parar. No hay ningún filtro en la placa: lo que se ve es lo
que el sensor entregó, muestra por muestra.

In [ ]:
df = dev.capture(3.0)

plt.plot(df['t'], df['y_uw'], color=AZUL)
plt.xlabel('t [s]'); plt.ylabel('ángulo [grados]')
plt.title('Girar el imán con la mano')
plt.show()

print(f'se movió {df["y_uw"].max() - df["y_uw"].min():.1f} grados')
print(f'ruido entre muestras consecutivas: {df["y_uw"].diff().std():.3f} grados '
      f'({df["y_uw"].diff().std() / 0.0879:.2f} cuentas)')


## 2. Configuración B: con actuador

Ahora el motor. El UNO no maneja un motor: maneja una llave, y la llave maneja al
motor con su propia fuente. Primero el caso completo --un puente en H, que acciona
en los dos sentidos-- y después, en 2.2, el de un solo cuadrante, que es el de
muchos bancos y al que le sirve casi todo lo de acá.

### 2.1 Un puente en H: los dos sentidos

```
   ┌───────────────┐                     ┌──────────────────┐
   │  Arduino UNO  │  9 ──── ENA ────────│      L298N       │      ┌─────────┐
   │               │  6 ──── IN1 ────────│   puente en H    │ OUT1─│  motor  │
   │  Timer1: PWM  │  7 ──── IN2 ────────│                  │ OUT2─│ + imán  │
   │               │ GND ─────┬──────────│ GND       +Vmot  │      └─────────┘
   └───────────────┘          │          └───────┬─────┬────┘
                              └── masa común ────┘     │
                                                       └── fuente del motor
                                                           (NO el USB)
```

`+Vmot` es la fuente del puente, y en este banco son 5 V --que es de donde sale
buena parte de lo que se cuenta más abajo.

**Tres reglas de cableado, y las tres se pagan caras si se saltean.** La fuente del
motor es propia y no el USB: el UNO da la lógica, nunca la potencia. Las masas van
unidas, porque si no las señales de control no tienen contra qué medirse. Y el
puente es el único que toca los bornes del motor: no hay ningún camino en el que un
pin del UNO vea la tensión de la fuente.

**El reparto de los tres pines no es arbitrario.** `ENA` lleva la *magnitud* por
PWM y el par `IN1`/`IN2` el *sentido*. Toda la modulación queda en un solo pin, y
el sentido en dos salidas digitales comunes. Lo que eso compra es que el sketch
pueda **apagar el puente antes de cambiar de sentido**: baja `ENA` --que abre las
cuatro llaves con una sola escritura--, recién entonces mueve `IN1` e `IN2`, y pasa
por el estado con las dos en bajo. Un cambio de sentido nunca atraviesa un estado
conduciendo. Con el comando en cero `ENA` queda en bajo, el puente abierto y el
motor en punto muerto: no frena el eje, sólo deja de empujarlo.

**El signo del ángulo lo deciden dos cables**: de qué lado están los del motor en
el puente, y de qué lado mira el imán al sensor. Si un comando positivo hace
*bajar* el ángulo medido, no hay nada roto: `ensayo.signo(df)` lo detecta sobre
cualquier captura con el motor andando, y `ensayo.normalizar()` lo aplica. O se
dan vuelta los dos cables del motor, que es exactamente lo mismo.

**El PWM va a 1 kHz, y es una concesión a este puente.** En abstracto conviene
modular más rápido: fuera del rango audible y con la ondulación de corriente lejos
de la banda del lazo. Pero el L298N es un puente de Darlington bipolares: cae unos
2 V entre sus dos lados --de 5 V de fuente al motor le llegan 2,5-- y tarda unos
2 us en conmutar. A 20 kHz, con períodos de 50 us, lo que se pierde en cada
transición se lleva una fracción grande de un tiempo de encendido que ya venía
escaso. **Medido en este banco: a 20 kHz el motor no arranca y a 1 kHz anda.**
Es `PWM_TOP` en el sketch; con un puente MOSFET --un TB6612FNG, un DRV8833, que
caen 0,3 V-- lo correcto sería subirla fuera del rango audible.

**Y la consecuencia que ordena todo lo demás: la zona muerta.** Con esos 2 V
comidos, por debajo de `u ≈ 60` el motor no arranca; a fondo, y sin carga, da 72
vueltas por segundo. Entre esas dos paredes vale un modelo lineal, y afuera no.
Nada de esto es del libro, y es lo que hace que valga la pena medirlo.

### 2.2 Un solo cuadrante: cuando el actuador empuja para un solo lado

Muy común, y no hace falta cambiar nada en el sketch: **alcanza con no mandar
comandos negativos.** Dos montajes llegan acá, y a los dos les sirve todo lo de
2.1 menos la parte del sentido:

```
   (a) el mismo L298N, cableado para un solo lado

   ┌───────────────┐                     ┌──────────────────┐
   │  Arduino UNO  │  9 ──── ENA ────────│      L298N       │      ┌─────────┐
   │               │                     │                  │ OUT1─│  motor  │
   │  Timer1: PWM  │    +5V ──── IN1 ────│ IN1 fijo en alto │ OUT2─│ + imán  │
   │               │    GND ──── IN2 ────│ IN2 fijo en bajo │      └─────────┘
   └───────────────┘                     └──────────────────┘


   (b) un transistor y un diodo: lo mínimo que mueve un motor

                       +Vmot
                         │
                  ┌──────┴──────┐
               [motor]     [diodo de retorno]      el cátodo (la banda) va
                  │             │                  del lado de +Vmot
                  └──────┬──────┘
                         │
   UNO 9 ──[1k]──► G ┌───┴────┐
                     │ MOSFET │  canal N, de puerta lógica, del lado de masa
                     └───┬────┘
                         │
                        GND ──── común con el UNO
```

En (b) **el diodo no es opcional**: al cortar la corriente, la inductancia del motor
la sigue empujando, y sin un camino de retorno esa energía aparece como un pico de
tensión sobre el transistor. Y una ventaja de (b) sobre el L298N: no hay 2 V de
caída, así que la zona muerta baja a lo que pida el rozamiento y el PWM puede ir
bien por encima del rango audible.

**Qué pasa con un `uff` negativo.** El sketch levanta la otra entrada de sentido y
modula igual. En (a) esa entrada no llega a ningún lado y el motor gira para el
mismo lado; en (b) no hay entrada de sentido y pasa lo mismo. Ninguna de las dos
cosas rompe nada, pero ninguna es un sentido: un banco de un cuadrante se usa con
comandos de 0 a 255 y listo.

**Y la polaridad se mira igual.** Si un comando positivo hace *bajar* el ángulo
medido, `ensayo.signo()` lo dice y `ensayo.normalizar()` lo da vuelta; el arreglo
físico son los dos cables del motor, o el imán dado vuelta. La celda de abajo lo
verifica con un empujón corto.

**Qué sigue funcionando completo**, que es todo lo que importa:

- **la sección 6 entera.** La curva estática y el escalón usan comandos positivos y
  nada más: la identificación **no necesita un puente bidireccional**.
- **la calibración del sensor** (`calibracion.ipynb`), que mide soltando el motor y
  dejándolo desacelerar por rozamiento. Soltar es justo lo que sabe hacer un
  cuadrante.
- Lo que no: la inversión de sentido, que es lo único que un cuadrante no tiene.


In [ ]:
dev.rest()
dev.uff = 120
dev.capture(0.4, warn=False)                      # que arranque
df = dev.capture(0.4, warn=False)                 # y ésta es la que se mira
dev.rest()

t, w = ensayo.velocidad(df, ventana=0)
s = ensayo.signo(df)
print(f'con u = +120 el eje fue a {np.mean(w):+.1f} rad/s: signo del banco {s:+d}')
if s > 0:
    print('  un comando positivo sube el ángulo, que es lo que tiene que ser')
else:
    print('  un comando positivo BAJA el ángulo: ensayo.normalizar() lo da vuelta solo;')
    print('  el arreglo físico son los dos cables del motor, o el imán dado vuelta')


### 2.3 La verificación completa

`bringup()`, ahora con el motor. Acciona el eje unas décimas de segundo y mira
que el ángulo se haya movido, y si hay sensor de corriente, que haya circulado
algo. Antes de eso mide el cero de la corriente con el puente abierto, que es la
única condición conocida que hay para calibrarlo.

Si el ángulo bajó con el comando positivo lo anota como una nota y no como una
falla, porque no lo es: es el signo del banco, y se aplica del lado de la
computadora.


In [ ]:
dev.bringup()                    # OJO: esto mueve el motor


La línea **`motor`** es la que cierra la cadena: salió un comando y volvió
movimiento, o corriente. Si no vuelve nada: alimentación del puente, `ENA`, o el
motor. `Puente_Bringup` separa «no sale el comando» de «el puente no lo sigue».

### El escalón

`uff` va directamente sobre el puente. Es la medición de la que sale todo lo
demás: contra esta respuesta se escribe cualquier modelo.

`dev.step()` mantiene `pre` segundos, cambia el parámetro y mantiene `post`
segundos más. La placa informa el tick exacto en el que cayó el cambio, así que
**`t = 0` es el escalón mismo con precisión de una muestra**, y la fluctuación de
temporización de esta computadora nunca entra en los datos.

In [ ]:
dev.rest()
dev.zero_current()               # el cero del sensor de corriente, puente abierto

df = dev.step('uff', 200, pre=0.3, post=0.9, back=0)
dev.rest()

t, w = ensayo.velocidad(df)

fig, (a, b, c) = plt.subplots(3, 1, sharex=True, figsize=(9, 6.4))
a.plot(t, w, color=AZUL);                      a.set_ylabel('velocidad [rad/s]')
b.plot(df['t'], df['u'], color=TENUE);         b.set_ylabel('u [cuentas de pwm]')
c.plot(df['t'], df['i'], color=NARANJA, lw=1); c.set_ylabel('i [mA]')
c.set_xlabel('t [s]')
for ax in (a, b, c):
    ax.axvline(0, color=TENUE, lw=0.9, ls='--')
a.set_title('Escalón: u = 0 → 200')
plt.show()

print(f'velocidad en régimen: {w[-1]:+.1f} rad/s ({w[-1] / (2 * np.pi):+.2f} vueltas/s) con u = +200')
if w[-1] < 0:
    print('el comando es positivo y el ángulo BAJA: es el signo del banco, '
          'y ensayo.normalizar() lo da vuelta')


## 3. Configuración C: sin medición de corriente

Es la configuración de la mayoría de los bancos, y no se pierde casi nada. Sin nada
conectado a A0:

- el canal `i` sigue apareciendo en cada fila, informando el ruido de una entrada
  al aire;
- `bringup()` **lo detecta y lo dice**: la entrada queda contra un riel del ADC, y
  eso es una *ausencia*, no un offset. La diferencia importa, porque calibrar un
  cero ahí dejaría un canal que informa ceros perfectos sin haber medido nada;
- lo que no se puede hacer es mirar el pico de arranque, ni usar la corriente
  como evidencia de que el motor está haciendo algo.

Todo el resto --el ángulo, la velocidad, la identificación de la planta
mecánica-- funciona igual. Si el sensor de corriente no está, la respuesta correcta
es seguir sin él.

## 4. Configuración D: con un ACS712

El ACS712 es un sensor de corriente de efecto Hall: la corriente atraviesa una
pista interna, el campo que genera se mide del otro lado de un aislamiento, y la
salida es una tensión analógica que va a A0.

**Dónde se lo inserta cambia lo que mide**, y las dos opciones son legítimas:

```
   (a) en la alimentación del puente -- unipolar

     fuente ──▶ IP+ [ACS712] IP- ──▶ +V del L298N
                      │
                     OUT ──▶ A0            mide el consumo del puente:
                     GND ──▶ GND común     los dos sentidos de giro salen POSITIVOS


   (b) en serie con un borne del motor -- bipolar

     OUT1 del L298N ──▶ IP+ [ACS712] IP- ──▶ motor
                              │
                             OUT ──▶ A0     mide la corriente del motor con signo,
                             GND ──▶ GND    pero flota con el PWM del puente
```

La (a) es la fácil y la que está en este banco. Su precio es que el signo de `i` no
es el signo del par: se ve igual girando para los dos lados, así que la corriente
**no sirve como evidencia del sentido**; por eso `bringup()` mira el ángulo.

**El ADC lee contra Vcc, y eso fija la resolución.** Un ACS712 alimentado a 5 V
reposa en 2,5 V para poder bajar cuando la corriente cambia de sentido, así que
contra la referencia interna de 1,1 V saturaría en reposo. Contra Vcc reposa en
media escala por construcción, y un LSB son 4,9 mV: con 185 mV/A, **26 mA por
cuenta**. Un motor chico consume decenas de mA en régimen, o sea una cuenta o
dos; lo que sí se ve es el arranque, que son cientos.

**Hay un cero que se puede medir y una ganancia que no.** El cero tiene una
condición conocida --con el puente abierto no circula corriente-- así que
`dev.zero_current()` lo mide y lo resta, y `bringup()` ya lo corre solo. Un cero
corrido se integra en toda medición posterior y no hay manera de conocerlo salvo
midiéndolo. La ganancia es otra cosa: haría falta una **corriente conocida**, y
desde el notebook no hay ninguna. Un tester en serie con el motor, una vez, alcanza;
el número vive en `SENSE_MV_PER_A`, en el sketch.

La celda que sigue mira la corriente de este banco y de paso diagnostica si hay algo
conectado en A0, con el mismo criterio que usa `bringup()`.

In [ ]:
dev.rest()
antes = dev.izero
dev.zero_current()
lsb = dev.channel('i').scale

df = dev.step('uff', 200, pre=0.3, post=0.9, back=0)
dev.rest()

# El diagnóstico se hace sobre el reposo --antes del escalón-- porque el criterio
# es el de bringup(): con el puente abierto no circula corriente, así que ahí la
# entrada tiene que estar lejos de los dos rieles del ADC y con margen para crecer.
# La placa cuenta en 12 bits sea cual sea su ADC.
adc = dev.izero + df[df['t'] < 0]['i'].mean() / lsb
al_aire = not (80 <= adc <= 4015)

plt.plot(df['t'], df['i'], color=NARANJA, lw=1)
plt.axvline(0, color=TENUE, lw=0.9, ls='--')
plt.axhline(0, color=TENUE, lw=0.9)
plt.xlabel('t [s]'); plt.ylabel('i [mA]')
plt.title('Corriente durante el escalón')
plt.show()

print(f'izero  {antes} → {dev.izero} cuentas del ADC     ({lsb:.1f} mA por cuenta)')
print(f'reposo {df[df["t"] < 0]["i"].mean():+.1f} mA,  '
      f'régimen {df[df["t"] > 0.4]["i"].mean():+.1f} mA,  '
      f'pico {df["i"].abs().max():.0f} mA')
print(f'ondulación en régimen: {df[df["t"] > 0.4]["i"].std() / lsb:.1f} cuentas RMS')
print()
if SIMULADO:
    print('el banco simulado no tiene ADC: este diagnóstico sólo dice algo con la placa')
elif al_aire:
    print(f'A0 parece estar AL AIRE: el reposo quedó en {adc:.0f} de 4095, contra un '
          f'riel del ADC. Eso es una ausencia y no un offset, así que no se calibra')
else:
    print(f'A0 reposa en {adc:.0f} de 4095: hay margen de +{(4095 - adc) * lsb / 1000:.1f} A '
          f'y -{adc * lsb / 1000:.1f} A para que la corriente crezca')


### ⁽¹⁾ Nota al pie: por qué esta medición no funciona del todo bien

Es la parte del banco que hay que mirar con desconfianza, y vale la pena que quede
escrito por qué. Ninguna de estas cosas es un cable flojo:

1. **La resolución no da para un motor chico.** 4,9 mV por cuenta contra 185 mV/A
   son 26 mA por cuenta, y el UNO además cuenta de a cuatro --su ADC es de 10 bits,
   corrido dos lugares para contar en 12 como el clon--. Un motor que consume 40 mA
   en régimen es una cuenta y media: lo que se ve es el ruido de cuantización con
   una señal abajo. El arranque, que son cientos de mA, sí se ve.

2. **Hay un cero que se puede medir y una ganancia que no.** El cero tiene una
   condición conocida --con el puente abierto no circula corriente-- y
   `zero_current()` lo mide. Para la ganancia haría falta una corriente conocida,
   y desde el notebook no hay ninguna. Hasta que alguien ponga un tester en serie,
   `SENSE_MV_PER_A` es una hipótesis. Y un divisor a la salida del sensor divide
   las dos cosas a la vez, el cero y la sensibilidad, así que el cero se calibra
   igual y la escala queda mintiendo en silencio.

3. **Este banco no reposa donde dice la hoja de datos.** Un ACS712 alimentado a 5 V
   reposa en 2,5 V; éste reposa en 3,75. Lo más probable es que el sensor esté
   alimentado con la fuente del puente y no con los 5 V del UNO, y entonces su
   salida puede pasar de 5 V, que es más de lo que le gusta a una entrada del AVR.
   Vale la pena confirmarlo con un tester.

4. **Se muestrea la ondulación del PWM en una fase fija.** El ADC convierte una vez
   por tick de 200 us y el PWM del puente va a 1 kHz: exactamente cinco muestras por
   período de modulación, y las dos frecuencias salen del mismo cristal de 16 MHz,
   así que la fase **no deriva nunca**. Lo que se mide no es el promedio de la
   corriente sino un puñado de instantes siempre en el mismo lugar de la rampa. El
   sesgo que eso introduce es fijo, y cambia con `dev.tickdiv`.

**Qué se puede hacer igual con este canal**: ver *cuándo* hay corriente, comparar
picos entre sí, ver el transitorio de arranque. Todo lo que sea relativo se
sostiene. Lo que no se sostiene es leer los mA como amperes de verdad.

Y así como está, **es un buen ejercicio de laboratorio**: medir la corriente con un
tester, comparar con lo que informa la placa, corregir `SENSE_MV_PER_A`, y volver a
medir.


## 5. La API

Todo lo de arriba se maneja con tres ideas: los parámetros son atributos, las
capturas devuelven un `DataFrame`, y las unidades son unidades reales.

### Conseguir el banco

```python
from bench import *

dev = sync_board()        # compila si cambió el sketch, graba si cambió el binario,
                          # y reabre el enlace (lo que resetea la placa)
dev = sync_board_cal()    # lo mismo, más la calibración del sensor de este banco
```

`sync_board()` no necesita que se le diga el puerto, y llamarlo al principio de
cada celda hace que cada una arranque desde los valores por omisión del sketch.
Este notebook usa `conseguir_banco()`, que hace lo mismo y **cae a un banco
simulado si no hay placa**, avisando fuerte que lo que se ve es un modelo. Es lo
que permite dar la clase con el cable desenchufado.

### Los parámetros son atributos

Se leen y se escriben como cualquier atributo, y cada uno viaja a la placa en el
momento. Son pocos:

| Atributo | Qué es |
|---|---|
| `uff` | el comando sobre el puente, de -255 a 255 |
| `tickdiv` | divisor del muestreador de 5 kHz: 10 → 500 Hz, 5 → 1 kHz, 50 → 100 Hz |
| `izero` | el cero de la corriente, en cuentas del ADC; `zero_current()` lo mide |
| `cal` | 1 si se aplica la tabla de calibración del sensor |
| `maxlate`, `missed`, `sovr`, `serr` | los contadores de salud del muestreo |
| `spres`, `mstat` | si el sensor contesta, y qué dice del imán |

No hay nada del lado de Python que los declare: la placa informa su propia tabla
de parámetros al conectarse, así que agregar un parámetro al sketch lo hace
aparecer acá solo.

### Lo que se hace con una captura

`ensayo.py` es el otro módulo, y es el que conviene leer entero: son cien líneas.

```python
import ensayo

t, w = ensayo.velocidad(df, ventana=0.02)   # rad/s: deriva y promedia 20 ms
s    = ensayo.signo(df)                     # +1 o -1: para dónde va el ángulo con u > 0
n    = ensayo.normalizar(df)                # t [s], u [%], theta [rad], omega [rad/s], i [A]
ensayo.guardar(df, 'datos/escalon.csv')     # lo mismo, a un archivo
df2  = ensayo.cargar('datos/escalon.csv')

dev.zero_current()        # la corriente de ahora es el cero (con el puente abierto)
dev.rest()                # comando en cero: donde termina todo experimento
dev.bringup()             # la verificación completa del equipo
```


In [ ]:
print(dev.info)
print()
print(f'{"dt":<10} {dev.dt * 1e3:.2f} ms por fila ({1 / dev.dt:.0f} Hz)')
print(f'{"tickdiv":<10} {dev.tickdiv}  (el muestreador va a 5 kHz)')
print()
print('las escalas que declara la placa, canal por canal:')
for nombre in ('y_uw', 'u', 'i'):
    c = dev.channel(nombre)
    print(f'  {c.name:<6} {c.scale:>10.4f} {c.unit} por cuenta')
print()
print('y lo que eso significa para la velocidad:')
dt = dev.dt
print(f'  una cuenta del sensor por fila son {np.deg2rad(360 / 4096) / dt:.2f} rad/s a {1 / dt:.0f} Hz')


### Capturar

`dev.capture(segundos)` emite y devuelve un `DataFrame` con una fila por período.
Las columnas son los canales de la placa, ya en unidades reales:

| Columna | Qué es |
|---|---|
| `t` | segundos desde el arranque de la captura, o desde el escalón si hubo uno |
| `y_raw` | el ángulo crudo del sensor, dentro de la vuelta y sin corregir |
| `y_uw` | el ángulo desenrollado, en grados |
| `u` | el comando que salió al puente, de -255 a 255 |
| `i` | la corriente, en mA |

Y en `df.attrs` viene **la salud de esa captura en particular**: los contadores se
ponen en cero antes de arrancar y se leen al terminar, así que hablan de estas filas
y no del historial de la placa. Toda captura además avisa por `stderr` si perdió
períodos o descartó filas, porque una serie temporal a la que le faltan muestras se
ve exactamente igual que una sana hasta que uno va a fijarse.

Tres formas de capturar, de menor a mayor:

```python
df = dev.capture(2.0)                                   # dos segundos y nada más
df = dev.capture(2.0, events=[(0.5, 'uff', 200)])       # con un cambio a los 0,5 s
df = dev.step('uff', 200, pre=0.3, post=0.7, back=0)    # un escalón, con t = 0 en el escalón
```

In [ ]:
df = dev.capture(1.0, warn=False)

print('columnas:', ', '.join(df.columns))
print(f'{len(df)} filas en {df.attrs["wall"]:.2f} s de reloj de pared')
print()
print('salud de esta captura:')
print(f'  períodos perdidos      {df.attrs["missed"]}')
print(f'  filas descartadas      {df.attrs["drops"]}')
print(f'  peor retardo           {df.attrs["maxlate"]} us de {df.attrs["dt_us"]:.0f} us '
      f'({df.attrs["maxlate"] / df.attrs["dt_us"]:.0%} del período)')
print(f'  desbordes del sensor   {df.attrs["sovr"]}')
print(f'  errores del bus i2c    {df.attrs["serr"]}')

## 6. Punto de partida para identificar la planta

Hasta acá el banco está descrito. Lo que sigue es el arranque de una práctica de
identificación: medir la planta con el lazo abierto, escribir un modelo, y
verificarlo prediciendo una medición que todavía no se hizo. Todo lo que sigue
usa comandos positivos y nada más, así que **corre igual con un actuador de un
solo cuadrante** (sección 2.2).

**Qué planta.** Lo que el comando mueve es la velocidad, no la posición. Un motor de
continua con carga inercial, visto desde el PWM, se parece a un primer orden:

$$\frac{\Omega(s)}{U(s)} = \frac{K}{1 + s\,\tau}$$

con `K` en radianes por segundo por cuenta de comando y `tau` dominada por la
inercia y el rozamiento (la constante eléctrica es mucho más rápida y queda
escondida). La posición es la integral de eso, así que vista desde el ángulo la
planta tiene un polo en el origen:

$$\frac{\Theta(s)}{U(s)} = \frac{K}{s\,(1 + s\,\tau)}$$

**Y dos cosas que no son lineales**, que en este banco no son un detalle: la zona
muerta de abajo (`|u| < 60`, el puente Darlington contra 5 V) y la saturación de
arriba (`|u| = 255`). Entre las dos, el modelo lineal vale en una ventana, y hay que
saber cuál es. Por eso conviene medir en este orden: primero la curva estática, que
dice dónde vale el modelo, y después el transitorio, que dice cuánto tarda.

### 6.1 La curva estática: la ganancia y la zona muerta

Un comando fijo, esperar el régimen, anotar la velocidad. Once veces.

In [ ]:
dev.rest()

comandos = np.arange(0, 251, 25)
medidas = []

for u in comandos:
    dev.uff = int(u)
    dev.capture(0.5, warn=False)                      # que llegue al régimen, y se tira
    df = dev.capture(0.4, warn=False)                 # ésta es la que cuenta

    # Sin promediar acá: el promedio de las diferencias sobre toda la ventana es
    # exactamente (último ángulo - primero) / tiempo, que es la velocidad media y
    # no le teme al ruido.
    _, w = ensayo.velocidad(df, ventana=0)
    medidas.append(np.mean(w))

dev.rest()

medidas = np.array(medidas)

# El signo es una propiedad del cableado, no de la planta: se lo saca del punto
# más rápido y se trabaja con la magnitud.
SIGNO = 1.0 if medidas[np.argmax(np.abs(medidas))] >= 0 else -1.0
vel = SIGNO * medidas

# La recta se ajusta sólo sobre la parte que se mueve: incluir la zona muerta en el
# ajuste es justo la manera de que la ganancia salga mal.
mueve = vel > 0.05 * vel.max()
K, b = np.polyfit(comandos[mueve], vel[mueve], 1)
u_muerto = -b / K

plt.plot(comandos, vel, 'o', color=AZUL, label='medido')
plt.plot(comandos[mueve], K * comandos[mueve] + b, color=NARANJA,
         label=f'recta: K = {K:.3f} (rad/s)/cuenta')
plt.axvline(u_muerto, color=TENUE, lw=0.9, ls='--')
plt.annotate(f'zona muerta: u ≈ {u_muerto:.0f}', (u_muerto, vel.max() * 0.5),
             xytext=(10, 0), textcoords='offset points', color=TENUE)
plt.xlabel('u [cuentas de pwm]'); plt.ylabel('velocidad en régimen [rad/s]')
plt.legend(); plt.title('Curva estática del actuador')
plt.show()

print(f'ganancia        K = {K:.4f} (rad/s) por cuenta de comando')
print(f'zona muerta     u ≈ {u_muerto:.0f} cuentas: por debajo de esto el eje no arranca')
print(f'a fondo         {vel.max():.1f} rad/s ({vel.max() / (2 * np.pi):.1f} vueltas/s) con u = {comandos[-1]}')
if SIGNO < 0:
    print('un comando positivo hace BAJAR el ángulo medido: es el signo del banco')


### 6.2 El escalón: la constante de tiempo

Ahora el transitorio, y **empezando por encima de la zona muerta**: un escalón que
arranca en cero mezcla el arranque con la dinámica que se quiere medir. De 100 a 220
las dos puntas están en la parte lineal de la curva de arriba.

`tau` se estima por el cruce del 63,2 % del salto, que es lo que se hace a mano
sobre una pantalla, y después se dibuja el primer orden encima para ver cuánto se
parece. Si el modelo no se apoya sobre la medición, el número no vale nada por más
prolijo que se vea.

In [ ]:
dev.rest()

U0, U1 = 100, 220

dev.uff = U0
dev.capture(0.6, warn=False)                           # que llegue al régimen de U0
esc = dev.step('uff', U1, pre=0.3, post=1.2, back=0)
dev.rest()

t, v = ensayo.velocidad(esc, ventana=0.02)
v = SIGNO * v

v0 = np.mean(v[(t > -0.25) & (t < -0.02)])             # régimen antes del escalón
v1 = np.mean(v[t > t.max() - 0.3])                     # régimen después
cruce = v0 + 0.632 * (v1 - v0)

# El primer cruce del 63,2 %, que es lo que uno hace a mano contra la pantalla, e
# interpolado entre las dos muestras que lo rodean. Buscar el cruce y no ajustar
# sobre toda la serie tiene una razón: después del régimen la velocidad se queda
# plana, y cualquier método que mire esa parte encuentra el nivel en todas partes.
t_post, v_post = t[t > 0], v[t > 0]
k = int(np.argmax(v_post >= cruce))
tau = (np.interp(cruce, v_post[k-1:k+1], t_post[k-1:k+1]) if k > 0 else t_post[0])

modelo = v0 + (v1 - v0) * (1 - np.exp(-np.clip(t, 0, None) / tau))

plt.plot(t, v, lw=1, color=NUBE, label='medido')
plt.plot(t, modelo, color=NARANJA, label=f'primer orden, tau = {tau*1e3:.0f} ms')
plt.axhline(cruce, color=TENUE, lw=0.9, ls=':')
plt.axvline(tau, color=TENUE, lw=0.9, ls='--')
plt.axvline(0, color=TENUE, lw=0.9)
plt.xlabel('t [s]'); plt.ylabel('velocidad [rad/s]'); plt.legend()
plt.title(f'Escalón de u = {U0} → {U1}')
plt.show()

K_esc = (v1 - v0) / (U1 - U0)
print(f'velocidad   {v0:.2f} → {v1:.2f} rad/s')
print(f'ganancia    K = {K_esc:.4f} (rad/s)/cuenta   '
      f'(la curva estática dio {K:.4f})')
print(f'constante   tau = {tau*1e3:.0f} ms   →   polo en {1/tau:.1f} rad/s')
print()
print(f'modelo:  Omega(s)/U(s) = {K_esc:.4f} / (1 + {tau:.3f} s)')
print(f'         Theta(s)/U(s) = {K_esc:.4f} / (s (1 + {tau:.3f} s))   [rad por cuenta]')


### 6.3 Verificar el modelo contra una medición que no se usó para ajustarlo

Un modelo ajustado sobre unos datos siempre se parece a esos datos. La pregunta es
si le acierta a **otro** escalón, uno que no participó del ajuste. Es la mitad del
trabajo que más se saltea y la única que decide si el modelo sirve.

In [ ]:
dev.rest()

V0, V1 = 120, 180                # un escalón distinto, dentro de la zona lineal

dev.uff = V0
dev.capture(0.6, warn=False)
val = dev.step('uff', V1, pre=0.2, post=1.0, back=0)
dev.rest()

tv, vv = ensayo.velocidad(val, ventana=0.02)
vv = SIGNO * vv

# La predicción sale SÓLO del modelo: el régimen de partida y de llegada salen de la
# curva estática, y la forma del tau del escalón anterior. No se mira la medición.
p0, p1 = K * V0 + b, K * V1 + b
pred = p0 + (p1 - p0) * (1 - np.exp(-np.clip(tv, 0, None) / tau))

plt.plot(tv, vv, lw=1, color=NUBE, label='medido')
plt.plot(tv, pred, color=AQUA, label='predicho por el modelo')
plt.axvline(0, color=TENUE, lw=0.9, ls='--')
plt.xlabel('t [s]'); plt.ylabel('velocidad [rad/s]'); plt.legend()
plt.title(f'Predicción contra medición: u = {V0} → {V1}')
plt.show()

err = vv[tv > 0] - pred[tv > 0]
print(f'error medio     {np.mean(err):+.2f} rad/s')
print(f'error RMS       {np.sqrt(np.mean(err**2)):.2f} rad/s, '
      f'sobre un salto de {p1 - p0:.2f}')


### 6.4 De acá en adelante

Lo de arriba es el piso: un primer orden y su verificación. Las preguntas que siguen
son las que vuelven interesante la práctica, y todas se contestan con las mismas
cuatro líneas de código cambiando un número:

- **¿`tau` depende del punto de operación?** Repetir 6.2 con escalones alrededor de
  u = 80, 150 y 230. Si `tau` cambia, el motor no es lineal en el rango, y el
  culpable habitual es el rozamiento: la fuerza que frena no crece proporcional a la
  velocidad.
- **¿Y de la dirección?** El mismo escalón con comandos negativos. Un banco simétrico
  da lo mismo; una diferencia grande apunta al puente o al montaje. Con un solo
  cuadrante la pregunta no aplica: no hay comandos negativos que aplicar.
- **¿Hay tiempo muerto?** Mirar de cerca los primeros milisegundos después del
  escalón. El sensor pone 0,3 ms, el muestreo hasta 2 más, y la inductancia del
  motor otro poco. Un primer orden puro no lo tiene; un modelo que lo lleve puede
  ajustar mejor, y conviene saber si eso es física o es el ajuste tapando algo.
- **La zona muerta como no linealidad.** Con `K` y `u_muerto` medidos, se puede
  prealimentar el salto: sumarle `u_muerto` con el signo del comando. Es lo que
  después cambia por completo el diseño de cualquier lazo cerrado.
- **La frecuencia de muestreo.** `dev.tickdiv = 5` lleva las filas a 1 kHz y `= 50`
  a 100 Hz. Qué le pasa a la velocidad calculada en cada caso, y por qué: una cuenta
  del sensor por fila es 0,77 rad/s a 500 Hz y 0,15 a 100.
- **Guardar cada ensayo.** `ensayo.guardar(df, 'datos/escalon_100_220.csv')` deja el
  archivo en las unidades en las que se escribe un modelo, y `ensayo.cargar()` lo
  trae de vuelta. Todo lo que viene después --ajustar, comparar, simular-- se hace
  sobre esos archivos y no sobre el banco.

Y dos advertencias que ya aparecieron y conviene repetir acá, porque son las que
arruinan una identificación sin dejar rastro:

**El sensor sin calibrar mete una ondulación de velocidad que parece del motor.**
Son unos veinticinco grados pico a pico en este banco, enganchados al ángulo y
repetidos vuelta tras vuelta. Al derivar salen como una oscilación de velocidad
perfectamente creíble. `calibracion.ipynb` la mide y la corrige.

**La velocidad se calcula acá, y cómo se calcula importa.** La placa entrega el
ángulo y nada más, sin ningún filtro: derivar amplifica el ruido, y la ventana de
`ensayo.velocidad()` es lo que lo tapa. Una ventana larga atenúa el ruido y redondea
el escalón; una corta muestra la cuantización desnuda. Las dos cosas se ven, y se
eligen, en este lado.
